In [1]:
import os
import re
import pandas as pd
import numpy as np

try:
    import pyarrow
    _has_parquet = True
except ImportError:
    _has_parquet = False

# Euro Stoxx 50 (EUSTX) Membership Tracking (2003–2026)

This notebook:
1. Loads the **t0 baseline** constituent list from `eustx_membership_2003-01-02.csv`
2. Parses ADD/DEL rebalance events from `EUSTX_changes` and applies them forward in time
3. Builds daily membership matrices (full and yfinance-only)
4. Combines individual ticker CSVs into `close_prices` matrix
5. Creates two tradable masks:
   - `tradable_mask_full.csv`: All tickers (for plotting constituent count)
   - `tradable_mask.csv`: Only yfinance-available tickers (for RL)

In [2]:
# Configuration
DATA_START_DATE = pd.Timestamp("2003-01-02")  # Start from yfinance data availability
DATA_END_DATE = pd.Timestamp("2026-03-15")    # End date for data

print(f"Building membership matrices from {DATA_START_DATE.date()} to {DATA_END_DATE.date()}")

Building membership matrices from 2003-01-02 to 2026-03-15


In [3]:
# Project paths
BASE = os.path.abspath(os.getcwd())
if os.path.basename(BASE) == "Data":
    BASE = os.path.dirname(BASE)

RAW_DIR = os.path.join(BASE, "Data", "Outputs", "Raw_Data")
RL_DIR = os.path.join(BASE, "Data", "Outputs", "RL_Needs")

for d in [RAW_DIR, RL_DIR]:
    os.makedirs(d, exist_ok=True)

print("RAW_DIR:", RAW_DIR)
print("RL_DIR:", RL_DIR)

RAW_DIR: /Users/kamilkashif/Documents/University/Masters Thesis/Master_Thesis_DRL_EUROSTOXX/Data/Outputs/Raw_Data
RL_DIR: /Users/kamilkashif/Documents/University/Masters Thesis/Master_Thesis_DRL_EUROSTOXX/Data/Outputs/RL_Needs


## STEP 1: Parse ADD/DEL Events from EUSTX_changes

In [4]:
BBG_EXCH_TO_YAHOO = {
    "NA": ".AS",
    "FP": ".PA",
    "GY": ".DE",
    "IM": ".MI",
    "SQ": ".MC",
    "SM": ".MC",
    "ID": ".IR",
    "FH": ".HE",
    "BB": ".BR",
}


def normalize_ticker(raw: str) -> str:
    """Bloomberg 'SYMBOL EX Equity' -> Yahoo Finance symbol (European listings)."""
    s = (raw or "").strip()
    if not s:
        return ""
    parts = s.split()
    if len(parts) < 2:
        return parts[0] if parts else ""
    symbol, exch = parts[0], parts[1].upper()
    if re.match(r"^\d{7,8}[A-Z]$", symbol.upper()):
        return symbol
    suf = BBG_EXCH_TO_YAHOO.get(exch)
    if suf:
        return f"{symbol}{suf}"
    return symbol


def extract_tickers_from_line(line: str) -> list:
    """Extract ticker symbols from a line. Handles +, -, *, • and multiple tickers."""
    tickers = []
    parts = re.split(r"[+*•\-]", line)
    for part in parts:
        part_clean = part.strip().lstrip("+*•").strip()
        if part_clean.startswith("-"):
            part_clean = part_clean.lstrip("-").strip()
        if not part_clean:
            continue
        t = normalize_ticker(part_clean)
        if t and t not in ("ADD", "DEL", "PERIOD", "TOTAL", "Equity", "COUNT"):
            tickers.append(t)
    if not tickers and line.strip():
        t = normalize_ticker(line.strip())
        if t and t not in ("ADD", "DEL", "PERIOD", "TOTAL", "Equity", "COUNT"):
            if not re.match(r"^\d+$", t):
                tickers.append(t)
    return tickers


def parse_eustx_log(filepath: str) -> pd.DataFrame:
    """Parse Euro Stoxx ADD/DEL log (EUSTX_changes) into events DataFrame."""
    if not os.path.isfile(filepath):
        raise FileNotFoundError(f"EUSTX_changes not found: {filepath}")
    with open(filepath, "r", encoding="utf-8", errors="replace") as f:
        lines = f.readlines()

    events = []
    current_date = None
    in_add = False
    in_del = False
    date_re = re.compile(r"(?:PERIOD\s*:\s*)?(\d{4}-\d{2}-\d{2})")

    for line in lines:
        line = line.strip()
        if not line:
            continue
        # Detect PERIOD / date-only line
        m = date_re.search(line)
        if m:
            current_date = m.group(1)
            in_add = False
            in_del = False
        if "ADD" in line.upper() and "DEL" not in line.upper():
            in_add = True
            in_del = False
            continue
        if "DEL" in line.upper() and "ADD" not in line.upper():
            in_del = True
            in_add = False
            continue
        if "TOTAL" in line.upper() or "PERIOD" in line.upper():
            continue
        if current_date is None:
            continue
        # Skip decorative separator lines (e.g. "-------------------------")
        if re.fullmatch(r"[\-\s_]{10,}", line):
            continue

        if in_add:
            for t in extract_tickers_from_line(line):
                events.append({"effective_date": current_date, "ticker": t, "action": "ADD"})
        elif in_del:
            for t in extract_tickers_from_line(line):
                events.append({"effective_date": current_date, "ticker": t, "action": "DEL"})

    df = pd.DataFrame(events)
    if df.empty:
        return df
    df["effective_date"] = pd.to_datetime(df["effective_date"])
    df = df.sort_values("effective_date").reset_index(drop=True)
    return df


def load_baseline_membership_csv(csv_path: str) -> set:
    """Load t0 Euro Stoxx 50 constituents from eustx_membership_2003-01-02.csv (Bloomberg-style rows)."""
    if not os.path.isfile(csv_path):
        raise FileNotFoundError(f"Baseline membership CSV not found: {csv_path}")
    df = pd.read_csv(csv_path)
    col = "ticker_bloomberg" if "ticker_bloomberg" in df.columns else df.columns[0]
    out = set()
    for raw in df[col].astype(str):
        t = normalize_ticker(raw.strip())
        if t:
            out.add(t)
    return out

In [5]:
# EUSTX_changes = rebalance history after t0; eustx_membership_2003-01-02.csv = t0 snapshot
DATA_DIR = os.path.join(BASE, "Data")
EUSTX_LOG = os.path.join(DATA_DIR, "EUSTX_changes")
if not os.path.isfile(EUSTX_LOG):
    EUSTX_LOG = os.path.join(BASE, "EUSTX_changes")

print(f"Parsing EUSTX_changes: {EUSTX_LOG}")
events = parse_eustx_log(EUSTX_LOG)


def _valid_eustx_ticker(t) -> bool:
    ts = str(t).strip()
    if not ts or set(ts) <= {"-"}:
        return False
    if re.fullmatch(r"-+", ts):
        return False
    return True


events = events[events["ticker"].map(_valid_eustx_ticker)].reset_index(drop=True)

# Save events (Euro Stoxx)
if _has_parquet:
    events.to_parquet(os.path.join(RAW_DIR, "eustx_events.parquet"), index=False)
events.to_csv(os.path.join(RAW_DIR, "eustx_events.csv"), index=False)

print(f"Parsed {len(events)} events")
if len(events):
    print(f"Date range: {events['effective_date'].min().date()} to {events['effective_date'].max().date()}")
print(f"\nFirst 20 events:")
print(events.head(20))

Parsing EUSTX_changes: /Users/kamilkashif/Documents/University/Masters Thesis/Master_Thesis_DRL_EUROSTOXX/Data/EUSTX_changes
Parsed 143 events
Date range: 2003-09-01 to 2025-10-01

First 20 events:
   effective_date   ticker action
0      2003-09-01   TIT.MI    ADD
1      2003-09-01    TI.MI    DEL
2      2003-10-01   IBE.MC    ADD
3      2003-10-01   HVM.DE    DEL
4      2004-08-01   SAP.DE    ADD
5      2004-08-01   AVE.PA    DEL
6      2004-10-01   ACA.PA    ADD
7      2004-10-01   VOW.DE    DEL
8      2005-07-01   TIM.MI    DEL
9      2005-07-01  AIBG.IR    ADD
10     2005-08-01   RNO.PA    ADD
11     2005-08-01   RDA.AS    DEL
12     2007-02-01   ISP.MI    ADD
13     2007-02-01   SPI.MI    DEL
14     2007-10-01    LG.PA    DEL
15     2007-10-01  AIBG.IR    DEL
16     2007-10-01    AD.AS    DEL
17     2007-10-01    SU.PA    ADD
18     2007-10-01   MTP.PA    ADD
19     2007-10-01    DG.PA    ADD


## STEP 2: Load yfinance Retrieved Tickers

In [6]:
# Load tickers that were successfully downloaded from yfinance
tickers_retrieved_path = os.path.join(RAW_DIR, "tickers_retrieved.csv")
tickers_not_retrieved_path = os.path.join(RAW_DIR, "tickers_not_retrieved.csv")

df_retrieved = pd.read_csv(tickers_retrieved_path)
df_not_retrieved = pd.read_csv(tickers_not_retrieved_path)

yfinance_tickers = sorted(df_retrieved['ticker'].tolist())
unavailable_tickers = sorted(df_not_retrieved['ticker'].tolist())

print(f"yfinance-available tickers: {len(yfinance_tickers)}")
print(f"Unavailable tickers: {len(unavailable_tickers)}")
print(f"  - Bloomberg IDs: {len(df_not_retrieved[df_not_retrieved['reason'] == 'Bloomberg ID'])}")
print(f"  - Delisted/Unavailable: {len(df_not_retrieved[df_not_retrieved['reason'] == 'Unavailable/Delisted'])}")

yfinance-available tickers: 79
Unavailable tickers: 23
  - Bloomberg IDs: 2
  - Delisted/Unavailable: 21


## STEP 3: Build Full Universe of Tickers

In [7]:
# Full universe = t0 baseline (CSV) ∪ events ∪ yfinance ∪ unavailable
BASELINE_CSV = os.path.join(DATA_DIR, "eustx_membership_2003-01-02.csv")
baseline_tickers = load_baseline_membership_csv(BASELINE_CSV)

event_tickers = set(events["ticker"].unique()) if len(events) else set()
all_tickers = sorted(
    event_tickers | set(yfinance_tickers) | set(unavailable_tickers) | baseline_tickers
)

print(f"\nFull universe: {len(all_tickers)} tickers")
print(f"  - Baseline t0 ({BASELINE_CSV}): {len(baseline_tickers)}")
print(f"  - Event tickers (EUSTX_changes): {len(event_tickers)}")
print(f"  - yfinance-available: {len(yfinance_tickers)}")
print(f"  - Unavailable: {len(unavailable_tickers)}")

# Identify Bloomberg-only IDs
bloomberg_only = [t for t in all_tickers if re.match(r"^\d{7,8}[A-Z]$", t)]
print(f"\nBloomberg IDs in full universe: {len(bloomberg_only)}")
print(f"Sample: {bloomberg_only[:10]}")


Full universe: 103 tickers
  - Baseline t0 (/Users/kamilkashif/Documents/University/Masters Thesis/Master_Thesis_DRL_EUROSTOXX/Data/eustx_membership_2003-01-02.csv): 50
  - Event tickers (EUSTX_changes): 85
  - yfinance-available: 79
  - Unavailable: 23

Bloomberg IDs in full universe: 2
Sample: ['1844030D', '3577044Z']


## STEP 4: Initial membership at t0

**t0** is `2003-01-02` (aligned with `DATA_START_DATE`). The full Euro Stoxx 50 constituent set at t0 is loaded from `eustx_membership_2003-01-02.csv`. Rebalances after t0 are applied via `EUSTX_changes` in `build_membership_daily`.

In [8]:
# Initial membership = t0 snapshot (CSV); EUSTX_changes applies subsequent ADD/DEL
START = DATA_START_DATE
END = DATA_END_DATE
FIRST_EVENT_DATE = events["effective_date"].min() if len(events) else None

initial_members = set(baseline_tickers)

print(f"Data range: {START.date()} to {END.date()}")
if FIRST_EVENT_DATE is not None:
    print(f"First rebalance in EUSTX_changes: {FIRST_EVENT_DATE.date()}")
else:
    print("First rebalance in EUSTX_changes: (no events)")

print(f"\nInitial members at t0 ({START.date()}): {len(initial_members)}")
print(f"Sample: {sorted(list(initial_members))[:20]}")

Data range: 2003-01-02 to 2026-03-15
First rebalance in EUSTX_changes: 2003-09-01

Initial members at t0 (2003-01-02): 50
Sample: ['1844030D', '3577044Z', 'AD.AS', 'AGN.AS', 'AI.PA', 'ALU.PA', 'ALV.DE', 'AVE.PA', 'BAS.DE', 'BAYN.DE', 'BBVA.MC', 'BN.PA', 'BNP.PA', 'CA.PA', 'CS.PA', 'DBK.DE', 'DTE.DE', 'ELE.MC', 'ENEL.MI', 'ENI.MI']


## STEP 5: Build membership_daily_full (ALL tickers)

In [9]:
def build_membership_daily(
    events: pd.DataFrame,
    start: pd.Timestamp,
    end: pd.Timestamp,
    all_tickers: list,
    initial_members: set,
) -> pd.DataFrame:
    """Build membership_daily matrix (0/1) from events.
    
    Logic:
    1. Initialize all tickers to 0
    2. Set initial_members to 1 for all dates
    3. Apply ADD/DEL events chronologically
    """
    date_index = pd.date_range(start=start, end=end, freq="D")
    membership = pd.DataFrame(0, index=date_index, columns=all_tickers, dtype=np.int8)
    membership.index.name = "date"

    # STEP 1: Set initial members to 1 for all dates
    for t in initial_members:
        if t in membership.columns:
            membership[t] = 1

    # STEP 2: Apply ADD/DEL events (these override the initial state)
    for _, row in events.iterrows():
        d = row["effective_date"]
        if isinstance(d, str):
            d = pd.Timestamp(d)
        t = row["ticker"]
        if t not in membership.columns:
            continue
        if row["action"] == "ADD":
            membership.loc[membership.index >= d, t] = 1
        else:  # DEL
            membership.loc[membership.index >= d, t] = 0

    membership = membership.clip(0, 1).fillna(0).astype(np.int8)
    return membership


# Build full membership matrix
print("Building membership_daily_full...")
membership_daily_full = build_membership_daily(
    events,
    start=START,
    end=END,
    all_tickers=all_tickers,
    initial_members=initial_members,
)

print(f"membership_daily_full shape: {membership_daily_full.shape}")
print(f"Date range: {membership_daily_full.index[0].date()} to {membership_daily_full.index[-1].date()}")

# Show membership count over time
member_counts = membership_daily_full.sum(axis=1)
print(f"\nMembership count statistics:")
print(member_counts.describe())
print(f"\nSample member counts:")
for d in [membership_daily_full.index[0], membership_daily_full.index[len(membership_daily_full)//2], membership_daily_full.index[-1]]:
    print(f"  {d.date()}: {membership_daily_full.loc[d].sum()} members")

Building membership_daily_full...
membership_daily_full shape: (8474, 103)
Date range: 2003-01-02 to 2026-03-15

Membership count statistics:
count    8474.000000
mean       49.720557
std         1.019709
min        46.000000
25%        50.000000
50%        50.000000
75%        50.000000
max        50.000000
dtype: float64

Sample member counts:
  2003-01-02: 50 members
  2014-08-09: 50 members
  2026-03-15: 46 members


In [10]:
for d in membership_daily_full.index:
    print(f"  {d.date()}: {membership_daily_full.loc[d].sum()} members")

  2003-01-02: 50 members
  2003-01-03: 50 members
  2003-01-04: 50 members
  2003-01-05: 50 members
  2003-01-06: 50 members
  2003-01-07: 50 members
  2003-01-08: 50 members
  2003-01-09: 50 members
  2003-01-10: 50 members
  2003-01-11: 50 members
  2003-01-12: 50 members
  2003-01-13: 50 members
  2003-01-14: 50 members
  2003-01-15: 50 members
  2003-01-16: 50 members
  2003-01-17: 50 members
  2003-01-18: 50 members
  2003-01-19: 50 members
  2003-01-20: 50 members
  2003-01-21: 50 members
  2003-01-22: 50 members
  2003-01-23: 50 members
  2003-01-24: 50 members
  2003-01-25: 50 members
  2003-01-26: 50 members
  2003-01-27: 50 members
  2003-01-28: 50 members
  2003-01-29: 50 members
  2003-01-30: 50 members
  2003-01-31: 50 members
  2003-02-01: 50 members
  2003-02-02: 50 members
  2003-02-03: 50 members
  2003-02-04: 50 members
  2003-02-05: 50 members
  2003-02-06: 50 members
  2003-02-07: 50 members
  2003-02-08: 50 members
  2003-02-09: 50 members
  2003-02-10: 50 members


In [11]:
# Save membership_daily_full
if _has_parquet:
    membership_daily_full.to_parquet(os.path.join(RL_DIR, "membership_daily_full.parquet"))
membership_daily_full.to_csv(os.path.join(RL_DIR, "membership_daily_full.csv"))
print(f"\nSaved membership_daily_full.csv: {membership_daily_full.shape}")


Saved membership_daily_full.csv: (8474, 103)


## STEP 6: Build membership_daily (yfinance-only)

In [12]:
# Filter to only yfinance-available tickers
yfinance_cols = [t for t in membership_daily_full.columns if t in yfinance_tickers]
membership_daily = membership_daily_full[yfinance_cols].copy()

print(f"membership_daily shape: {membership_daily.shape}")
print(f"Filtered from {membership_daily_full.shape[1]} to {membership_daily.shape[1]} tickers")

# Show membership count for yfinance-only
member_counts_yf = membership_daily.sum(axis=1)
print(f"\nyfinance membership count statistics:")
print(member_counts_yf.describe())
print(f"\nSample member counts:")
for d in [membership_daily.index[0], membership_daily.index[len(membership_daily)//2], membership_daily.index[-1]]:
    print(f"  {d.date()}: {membership_daily.loc[d].sum()} tradable members")

membership_daily shape: (8474, 79)
Filtered from 103 to 79 tickers

yfinance membership count statistics:
count    8474.000000
mean       45.866415
std         2.753768
min        38.000000
25%        45.000000
50%        47.000000
75%        48.000000
max        49.000000
dtype: float64

Sample member counts:
  2003-01-02: 38 tradable members
  2014-08-09: 47 tradable members
  2026-03-15: 45 tradable members


In [13]:
# Save membership_daily (yfinance-only)
if _has_parquet:
    membership_daily.to_parquet(os.path.join(RL_DIR, "membership_daily.parquet"))
membership_daily.to_csv(os.path.join(RL_DIR, "membership_daily.csv"))
print(f"\nSaved membership_daily.csv: {membership_daily.shape}")


Saved membership_daily.csv: (8474, 79)


## STEP 7: Build close_prices from Individual Ticker Files

In [14]:
import glob

# Find all ticker CSV files (exclude special files)
SPECIAL_FILES = {'close_prices.csv', 'close_prices.parquet',
                 'ndx_events.csv', 'ndx_events.parquet',
                 'eustx_events.csv', 'eustx_events.parquet',
                 'nky_events.csv', 'nky_events.parquet',
                 'tickers_retrieved.csv', 'tickers_not_retrieved.csv'}
ticker_files = [f for f in glob.glob(os.path.join(RAW_DIR, "*.csv"))
                if os.path.basename(f) not in SPECIAL_FILES]

print(f"Found {len(ticker_files)} ticker CSV files in {RAW_DIR}")

# Build close_prices by reading each ticker file
close_data = {}
missing_tickers = []
for fpath in ticker_files:
    ticker = os.path.basename(fpath).replace('.csv', '')
    try:
        df = pd.read_csv(fpath)
        if 'close' in df.columns and 'date' in df.columns:
            # Parse date
            df['date'] = pd.to_datetime(df['date'])
            # Set date as index and extract close prices
            series = df.set_index('date')['close']
            close_data[ticker] = series
        else:
            missing_tickers.append(ticker)
    except Exception as e:
        print(f"  Error reading {ticker}: {e}")
        missing_tickers.append(ticker)

print(f"Successfully loaded close prices for {len(close_data)} tickers")
if missing_tickers:
    print(f"Missing/failed tickers: {missing_tickers[:10]}{'...' if len(missing_tickers) > 10 else ''}")

# Combine into a single DataFrame
close_prices = pd.DataFrame(close_data)
close_prices = close_prices.sort_index()
print(f"\nclose_prices shape: {close_prices.shape}")
print(f"close_prices date range: {close_prices.index.min().date()} to {close_prices.index.max().date()}")

Found 79 ticker CSV files in /Users/kamilkashif/Documents/University/Masters Thesis/Master_Thesis_DRL_EUROSTOXX/Data/Outputs/Raw_Data
Successfully loaded close prices for 79 tickers

close_prices shape: (5961, 79)
close_prices date range: 2003-01-02 to 2026-03-13


In [15]:
# Save close_prices
CLOSE_PATH = os.path.join(RL_DIR, "close_prices.parquet")
CLOSE_CSV = os.path.join(RL_DIR, "close_prices.csv")
if _has_parquet:
    close_prices.to_parquet(CLOSE_PATH)
close_prices.to_csv(CLOSE_CSV)
print(f"Saved close_prices to {CLOSE_CSV}")

Saved close_prices to /Users/kamilkashif/Documents/University/Masters Thesis/Master_Thesis_DRL_EUROSTOXX/Data/Outputs/RL_Needs/close_prices.csv


## STEP 8: Build tradable_mask_full and tradable_mask

Tradable = (in Euro Stoxx 50 index) AND (has price data)

In [16]:
# Align membership_daily_full with close_prices dates
# Reindex membership to match close_prices date range
membership_aligned = membership_daily_full.reindex(close_prices.index).fillna(0).astype(np.int8)

# Align close_prices to have all tickers from membership
all_cols = sorted(set(membership_aligned.columns) | set(close_prices.columns))
mh_full = membership_aligned.reindex(columns=all_cols).fillna(0).clip(0, 1).astype(np.int8)
cp_full = close_prices.reindex(columns=all_cols)

print(f"Aligned shapes:")
print(f"  membership: {mh_full.shape}")
print(f"  close_prices: {cp_full.shape}")
print(f"  Date range: {mh_full.index[0].date()} to {mh_full.index[-1].date()}")

Aligned shapes:
  membership: (5961, 103)
  close_prices: (5961, 103)
  Date range: 2003-01-02 to 2026-03-13


In [17]:
# tradable_mask_full = (member) AND (has price)
tradable_mask_full = ((mh_full == 1) & cp_full.notna()).astype(np.int8)

if _has_parquet:
    tradable_mask_full.to_parquet(os.path.join(RL_DIR, "tradable_mask_full.parquet"))
tradable_mask_full.to_csv(os.path.join(RL_DIR, "tradable_mask_full.csv"))

print(f"\ntradable_mask_full shape: {tradable_mask_full.shape}")
print(f"Saved tradable_mask_full.csv")

# Show tradable counts
print(f"\nTradable count at sample dates:")
for ts in [tradable_mask_full.index[0], tradable_mask_full.index[len(tradable_mask_full)//2], tradable_mask_full.index[-1]]:
    mem_count = int(mh_full.loc[ts].sum())
    trad_count = int(tradable_mask_full.loc[ts].sum())
    print(f"  {ts.date()}: {trad_count} tradable / {mem_count} members")


tradable_mask_full shape: (5961, 103)
Saved tradable_mask_full.csv

Tradable count at sample dates:
  2003-01-02: 37 tradable / 50 members
  2014-07-25: 47 tradable / 50 members
  2026-03-13: 45 tradable / 46 members


In [18]:
# tradable_mask (yfinance-only)
yfinance_cols_available = [t for t in yfinance_tickers if t in tradable_mask_full.columns]
tradable_mask = tradable_mask_full[yfinance_cols_available].copy()

if _has_parquet:
    tradable_mask.to_parquet(os.path.join(RL_DIR, "tradable_mask.parquet"))
tradable_mask.to_csv(os.path.join(RL_DIR, "tradable_mask.csv"))

print(f"\ntradable_mask shape: {tradable_mask.shape}")
print(f"Saved tradable_mask.csv (yfinance-only, for RL)")

# Show tradable counts for yfinance-only
print(f"\nTradable count (yfinance-only) at sample dates:")
for ts in [tradable_mask.index[0], tradable_mask.index[len(tradable_mask)//2], tradable_mask.index[-1]]:
    trad_count = int(tradable_mask.loc[ts].sum())
    print(f"  {ts.date()}: {trad_count} tradable")


tradable_mask shape: (5961, 79)
Saved tradable_mask.csv (yfinance-only, for RL)

Tradable count (yfinance-only) at sample dates:
  2003-01-02: 37 tradable
  2014-07-25: 47 tradable
  2026-03-13: 45 tradable


## Sanity Checks

In [19]:
print("=" * 60)
print("SANITY CHECKS")
print("=" * 60)

# Check 1: No tradable where close is NaN
bad = (tradable_mask_full == 1) & cp_full.isna()
assert bad.sum().sum() == 0, "tradable_mask_full has 1 where close is NaN"
print("✓ No tradable_mask=1 where close is NaN")

# Check 2: membership value range
assert membership_daily_full.min().min() == 0
assert membership_daily_full.max().max() == 1
print("✓ membership_daily_full values are 0 or 1")

# Check 3: tradable_mask value range
assert tradable_mask_full.min().min() == 0
assert tradable_mask_full.max().max() == 1
print("✓ tradable_mask_full values are 0 or 1")

# Check 4: Date ranges match
assert membership_daily_full.index[0].date() == START.date()
assert membership_daily_full.index[-1].date() == END.date()
print(f"✓ Date range is {START.date()} to {END.date()}")

# Check 5: Member count should be 50 (Euro Stoxx 50)
member_counts = membership_daily_full.sum(axis=1)
avg_members = member_counts.mean()
print(f"✓ Average members per day: {avg_members:.1f} (expected ~50)")

print("\n" + "=" * 60)
print("ALL SANITY CHECKS PASSED")
print("=" * 60)

SANITY CHECKS
✓ No tradable_mask=1 where close is NaN
✓ membership_daily_full values are 0 or 1
✓ tradable_mask_full values are 0 or 1
✓ Date range is 2003-01-02 to 2026-03-15
✓ Average members per day: 49.7 (expected ~50)

ALL SANITY CHECKS PASSED


## Summary

In [20]:
print("\n" + "=" * 60)
print("SUMMARY")
print("=" * 60)
print(f"\nData range: {START.date()} to {END.date()}")
print(f"Total days: {len(membership_daily_full)}")
print(f"\nTickers:")
print(f"  - Full universe: {len(all_tickers)} tickers")
print(f"  - yfinance-available: {len(yfinance_tickers)} tickers")
print(f"  - Unavailable: {len(unavailable_tickers)} tickers")
print(f"\nFiles created:")
print(f"  - membership_daily_full.csv: {membership_daily_full.shape}")
print(f"  - membership_daily.csv: {membership_daily.shape}")
print(f"  - close_prices.csv: {close_prices.shape}")
print(f"  - tradable_mask_full.csv: {tradable_mask_full.shape}")
print(f"  - tradable_mask.csv: {tradable_mask.shape}")
print(f"\nAll files saved to:")
print(f"  - Raw data: {RAW_DIR}")
print(f"  - RL matrices: {RL_DIR}")
print("\n" + "=" * 60)
print("COMPLETE!")
print("=" * 60)


SUMMARY

Data range: 2003-01-02 to 2026-03-15
Total days: 8474

Tickers:
  - Full universe: 103 tickers
  - yfinance-available: 79 tickers
  - Unavailable: 23 tickers

Files created:
  - membership_daily_full.csv: (8474, 103)
  - membership_daily.csv: (8474, 79)
  - close_prices.csv: (5961, 79)
  - tradable_mask_full.csv: (5961, 103)
  - tradable_mask.csv: (5961, 79)

All files saved to:
  - Raw data: /Users/kamilkashif/Documents/University/Masters Thesis/Master_Thesis_DRL_EUROSTOXX/Data/Outputs/Raw_Data
  - RL matrices: /Users/kamilkashif/Documents/University/Masters Thesis/Master_Thesis_DRL_EUROSTOXX/Data/Outputs/RL_Needs

COMPLETE!


In [21]:
import yfinance as yf
import pandas as pd
import os

# Setup path
RL_DIR = os.path.join(BASE, "Data", "Outputs", "RL_Needs")
os.makedirs(RL_DIR, exist_ok=True)

# Download QQQ
qqq = yf.download('FEZ', start='2003-01-02', end='2026-03-15', interval='1d', progress=False)
if not qqq.empty:
    # Flatten MultiIndex columns if present
    if isinstance(qqq.columns, pd.MultiIndex):
        qqq.columns = qqq.columns.get_level_values(0)
    qqq = qqq.reset_index()
    qqq.columns = [c.lower() for c in qqq.columns]
    qqq.to_csv(os.path.join(RL_DIR, 'QQQ.csv'), index=False)
    print(f"Saved QQQ.csv: {len(qqq)} rows")

Saved QQQ.csv: 5836 rows


In [22]:
# Note: ^FVX is 5-year Treasury yield (closest available on Yahoo Finance)
risk_free = yf.download('^FVX', start='2003-01-02', end='2026-03-15', interval='1d', progress=False)
if not risk_free.empty:
    if isinstance(risk_free.columns, pd.MultiIndex):
        risk_free.columns = risk_free.columns.get_level_values(0)
    risk_free = risk_free.reset_index()
    risk_free.columns = [c.lower() for c in risk_free.columns]
    risk_free.to_csv(os.path.join(RL_DIR, 'risk_free_data.csv'), index=False)
    print(f"Saved risk_free_data.csv: {len(risk_free)} rows")

Saved risk_free_data.csv: 5830 rows


In [23]:
# Download VIX
vix = yf.download('^VIX', start='2003-01-02', end='2026-03-15', interval='1d', progress=False)
if not vix.empty:
    if isinstance(vix.columns, pd.MultiIndex):
        vix.columns = vix.columns.get_level_values(0)
    vix = vix.reset_index()
    vix.columns = [c.lower() for c in vix.columns]
    vix.to_csv(os.path.join(RL_DIR, 'VIX.csv'), index=False)
    print(f"Saved VIX.csv: {len(vix)} rows")

Saved VIX.csv: 5836 rows


In [24]:
treasury = yf.download('^IRX', start='2003-01-02', end='2026-03-15', interval='1d', progress=False)
if not treasury.empty:
    if isinstance(treasury.columns, pd.MultiIndex):
        treasury.columns = treasury.columns.get_level_values(0)
    treasury = treasury.reset_index()
    treasury.columns = [c.lower() for c in treasury.columns]
    treasury.to_csv(os.path.join(RL_DIR, 'risk_free_data.csv'), index=False)
    print(f"✓ Saved risk_free_data.csv: {len(treasury)} rows from {treasury['date'].min()} to {treasury['date'].max()}")
else:
    print("✗ Failed to download Treasury data")

✓ Saved risk_free_data.csv: 5830 rows from 2003-01-02 00:00:00 to 2026-03-13 00:00:00


In [25]:
import os
import pandas as pd

# Base paths (same logic as in your other notebooks)
BASE = os.path.abspath(os.getcwd())
if os.path.basename(BASE) == "Data":
    BASE = os.path.dirname(BASE)

OUT_DATA_DIR = os.path.join(BASE, "Data", "Outputs", "RL_Needs")
print("Using OUT_DATA_DIR:", OUT_DATA_DIR)

def load_and_describe(name, filename, n_head=5):
    path = os.path.join(OUT_DATA_DIR, filename)
    print("\n" + "="*80)
    print(f"{name}  ({filename})")
    print("="*80)
    df = pd.read_csv(path, parse_dates=[0])
    print(f"Shape: {df.shape}")
    print(f"Columns ({len(df.columns)}): {list(df.columns)[:10]}{' ...' if len(df.columns) > 10 else ''}")
    print(f"Date range: {df.iloc[0,0]}  →  {df.iloc[-1,0]}")
    print(f"\nHead (first {n_head} rows):")
    print(df.head(n_head))
    return df

# 1) RL core matrices
df_membership_daily = load_and_describe(
    "membership_daily (RL universe, yfinance-only)",
    "membership_daily.csv",
)

df_tradable_mask = load_and_describe(
    "tradable_mask (RL tradable universe, yfinance-only)",
    "tradable_mask.csv",
)

# 2) Prices matrix
df_close_prices = load_and_describe(
    "close_prices (per-ticker daily close, RL universe)",
    "close_prices.csv",
)

# 3) Market-wide series (QQQ.csv holds FEZ — Eurozone large-cap ETF proxy)
df_QQQ = load_and_describe(
    "QQQ.csv (FEZ Euro Stoxx ETF proxy)",
    "QQQ.csv",
)

df_VIX = load_and_describe(
    "VIX (CBOE Volatility Index)",
    "VIX.csv",
)

df_risk_free = load_and_describe(
    "risk_free_data (short-rate / Treasury proxy)",
    "risk_free_data.csv",
)

print("\n" + "="*80)
print("SUMMARY")
print("="*80)
print(f"membership_daily:          {df_membership_daily.shape}")
print(f"tradable_mask:             {df_tradable_mask.shape}")
print(f"close_prices:              {df_close_prices.shape}")
print(f"QQQ:                       {df_QQQ.shape}")
print(f"VIX:                       {df_VIX.shape}")
print(f"risk_free_data:            {df_risk_free.shape}")

Using OUT_DATA_DIR: /Users/kamilkashif/Documents/University/Masters Thesis/Master_Thesis_DRL_EUROSTOXX/Data/Outputs/RL_Needs

membership_daily (RL universe, yfinance-only)  (membership_daily.csv)
Shape: (8474, 80)
Columns (80): ['date', 'ABI.BR', 'ACA.PA', 'AD.AS', 'ADS.DE', 'ADYEN.AS', 'AGN.AS', 'AGS.BR', 'AI.PA', 'AIR.PA'] ...
Date range: 2003-01-02 00:00:00  →  2026-03-15 00:00:00

Head (first 5 rows):
        date  ABI.BR  ACA.PA  AD.AS  ADS.DE  ADYEN.AS  AGN.AS  AGS.BR  AI.PA  \
0 2003-01-02       0       0      1       0         0       1       0      1   
1 2003-01-03       0       0      1       0         0       1       0      1   
2 2003-01-04       0       0      1       0         0       1       0      1   
3 2003-01-05       0       0      1       0         0       1       0      1   
4 2003-01-06       0       0      1       0         0       1       0      1   

   AIR.PA  ...  TEF.MC  TIT.MI  TTE.PA  UCG.MI  UMG.AS  VIV.PA  VNA.DE  \
0       0  ...       1       0      